In [1]:
# ==========================================
# 0) Install + Imports + Paths
# ==========================================
!pip -q install pydicom timm einops opencv-python tqdm pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg gdcm

import os, gc, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

CLASSIFY_ROOT = Path("/kaggle/input/datasets/chonlanawawa/bloodclot-classify/bloodclot_classify/bloodclot_classify")
LABELS_CSV = CLASSIFY_ROOT / "labels_slices.csv"
assert LABELS_CSV.exists(), LABELS_CSV

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 82.0 MB/s eta 0:00:00
device: cuda


In [2]:
# ==========================================
# 1) DICOM -> HU -> multi-window preprocessing
# ==========================================
WINDOWS = [(40, 80), (80, 200), (600, 2800)]  # brain / subdural-ish / bone

def dicom_to_hu(ds: pydicom.Dataset) -> np.ndarray:
    img = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    img = img * slope + intercept
    phot = str(getattr(ds, "PhotometricInterpretation", "")).upper()
    if phot == "MONOCHROME1":
        img = img.max() - img
    return img

def apply_window(img_hu: np.ndarray, center: float, width: float) -> np.ndarray:
    lo = center - width/2
    hi = center + width/2
    x = np.clip(img_hu, lo, hi)
    x = (x - lo) / (hi - lo + 1e-6)
    return x.astype(np.float32)

def read_instance_number(path: str) -> int:
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        return int(getattr(ds, "InstanceNumber", 10**9))
    except Exception:
        return 10**9

def preprocess_dicom(path: str, out_size=256) -> np.ndarray:
    ds = pydicom.dcmread(path, force=True)
    hu = dicom_to_hu(ds)
    chans = []
    for c, w in WINDOWS:
        x = apply_window(hu, c, w)
        x = cv2.resize(x, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
        chans.append(x)
    return np.stack(chans, axis=0)  # (C,H,W)

def preprocess_dicom_safe(path: str, out_size=256) -> np.ndarray:
    try:
        return preprocess_dicom(path, out_size)
    except Exception:
        return np.zeros((len(WINDOWS), out_size, out_size), dtype=np.float32)

In [3]:
# ==========================================
# 2) Build study-level table
# ==========================================
df = pd.read_csv(LABELS_CSV)
df["abs_path"] = df["dicom_relpath"].apply(lambda p: str(CLASSIFY_ROOT / p))

group_cols = ["dataset", "study_uid", "series_uid", "split"]
gb = df.groupby(group_cols)
total_groups = gb.ngroups

studies = []
print(f"Index studies: total_groups={total_groups}")
t0 = time.time()

for i, (key, sub) in enumerate(gb, 1):
    dataset, study_uid, series_uid, split = key
    paths = sub["abs_path"].tolist()
    paths = sorted(paths, key=read_instance_number)

    studies.append({
        "dataset": dataset,
        "study_uid": study_uid,
        "series_uid": series_uid,
        "split": split,
        "paths": paths,
        "y_hemo": int(sub["has_hem"].max()),
        "y_isc": int(sub["has_isc"].max()),
    })

    if i == 1 or i % 500 == 0 or i == total_groups:
        dt = time.time() - t0
        rate = i / max(dt, 1e-6)
        eta = (total_groups - i) / max(rate, 1e-6)
        print(f"[Index studies] {i}/{total_groups} ({100*i/total_groups:.1f}%) rate={rate:.2f}/s ETA={eta/60:.1f} min")

studies_df = pd.DataFrame(studies)
print("studies_df:", studies_df.shape)

Index studies: total_groups=10213
[Index studies] 1/10213 (0.0%) rate=2.79/s ETA=61.0 min
[Index studies] 500/10213 (4.9%) rate=3.72/s ETA=43.5 min
[Index studies] 1000/10213 (9.8%) rate=2.90/s ETA=52.9 min
[Index studies] 1500/10213 (14.7%) rate=2.76/s ETA=52.7 min
[Index studies] 2000/10213 (19.6%) rate=2.46/s ETA=55.7 min
[Index studies] 2500/10213 (24.5%) rate=2.70/s ETA=47.5 min
[Index studies] 3000/10213 (29.4%) rate=3.19/s ETA=37.7 min
[Index studies] 3500/10213 (34.3%) rate=3.66/s ETA=30.6 min
[Index studies] 4000/10213 (39.2%) rate=4.11/s ETA=25.2 min
[Index studies] 4500/10213 (44.1%) rate=4.55/s ETA=20.9 min
[Index studies] 5000/10213 (49.0%) rate=4.98/s ETA=17.5 min
[Index studies] 5500/10213 (53.9%) rate=5.39/s ETA=14.6 min
[Index studies] 6000/10213 (58.7%) rate=5.79/s ETA=12.1 min
[Index studies] 6500/10213 (63.6%) rate=6.18/s ETA=10.0 min
[Index studies] 7000/10213 (68.5%) rate=6.55/s ETA=8.2 min
[Index studies] 7500/10213 (73.4%) rate=6.92/s ETA=6.5 min
[Index studies]

In [4]:
# ==========================================
# 3) Dataset
# ==========================================
class CTStudyDataset(Dataset):
    def __init__(self, df: pd.DataFrame, label_col: str, n_slices=32, out_size=256, augment=False):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col
        self.n_slices = n_slices
        self.out_size = out_size
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _sample_idx(self, L: int) -> np.ndarray:
        if L <= self.n_slices:
            idx = np.arange(L)
            if L < self.n_slices:
                pad = np.random.choice(idx, size=self.n_slices - L, replace=True)
                idx = np.concatenate([idx, pad])
            return idx
        return np.linspace(0, L-1, num=self.n_slices, dtype=int)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        paths = row["paths"]
        idx = self._sample_idx(len(paths))

        xs = []
        for j in idx:
            x = preprocess_dicom_safe(paths[int(j)], out_size=self.out_size)

            if self.augment:
                if random.random() < 0.5:
                    x = x[:, :, ::-1].copy()
                if random.random() < 0.2:
                    noise = np.random.normal(0, 0.01, size=x.shape).astype(np.float32)
                    x = np.clip(x + noise, 0, 1)

            xs.append(x)

        x = torch.tensor(np.stack(xs, 0), dtype=torch.float32)  # (N,C,H,W)
        y = torch.tensor([float(row[self.label_col])], dtype=torch.float32)
        return x, y

In [5]:
# ==========================================
# 4) Model (CNN + Transformer)
# ==========================================
class SliceEncoder(nn.Module):
    def __init__(self, backbone="tf_efficientnetv2_s", pretrained=True, out_dim=512):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="avg")
        in_dim = self.backbone.num_features
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        feat = self.backbone(x)
        return self.proj(feat)

class StudyTransformer(nn.Module):
    def __init__(self, backbone="tf_efficientnetv2_s", pretrained=True, emb_dim=512, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.encoder = SliceEncoder(backbone, pretrained, out_dim=emb_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))

        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, dim_feedforward=emb_dim*4,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(emb_dim), nn.Linear(emb_dim, 1))

    def forward(self, x):
        B,N,C,H,W = x.shape
        feat = self.encoder(x.view(B*N, C, H, W)).view(B, N, -1)
        cls = self.cls_token.expand(B, 1, feat.size(-1))
        tok = torch.cat([cls, feat], dim=1)
        tok = self.transformer(tok)
        return self.head(tok[:,0])

In [6]:
# ==========================================
# 5) Loss: Focal+BCE (good for Stage 2)
# ==========================================
class FocalBCEWithLogits(nn.Module):
    def __init__(self, pos_weight=None, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )
        p = torch.sigmoid(logits)
        pt = torch.where(targets == 1, p, 1 - p)
        focal = (1 - pt).pow(self.gamma) * bce

        if self.alpha is not None:
            alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
            focal = alpha_t * focal

        if self.reduction == "mean":
            return focal.mean()
        if self.reduction == "sum":
            return focal.sum()
        return focal

In [7]:
# ==========================================
# 6) Train/Eval loops (loss + accuracy)
# ==========================================
def train_one_epoch(model, loader, opt, device, epoch=1, crit=None, desc="train",
                    print_every=100, use_tqdm=True, thr=0.5):
    model.train()
    if crit is None:
        crit = nn.BCEWithLogitsLoss()

    running_loss, seen = 0.0, 0
    running_correct = 0.0

    it = loader
    if use_tqdm:
        it = tqdm(loader, desc=f"[{desc}] ep{epoch}", leave=True, dynamic_ncols=True)

    t0 = time.time()
    for step, (x, y) in enumerate(it, 1):
        x, y = x.to(device), y.to(device)

        opt.zero_grad(set_to_none=True)
        logit = model(x)
        loss = crit(logit, y)
        loss.backward()
        opt.step()

        bs = x.size(0)
        running_loss += loss.item() * bs
        seen += bs

        probs = torch.sigmoid(logit)
        preds = (probs >= thr).float()
        running_correct += (preds == y).float().sum().item()

        avg_loss = running_loss / max(seen, 1)
        avg_acc = running_correct / max(seen, 1)

        if use_tqdm:
            it.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.3f}")

        if step == 1 or step % print_every == 0 or step == len(loader):
            dt = time.time() - t0
            ips = seen / max(dt, 1e-6)
            print(f"[{desc}] ep={epoch} step={step}/{len(loader)} "
                  f"avg_loss={avg_loss:.6f} acc={avg_acc:.4f} thr={thr:.2f} ips={ips:.2f}")

    return avg_loss, avg_acc

@torch.no_grad()
def eval_one_epoch(model, loader, device, epoch=1, crit=None, desc="val",
                   print_every=200, use_tqdm=True, thr=0.5):
    model.eval()
    if crit is None:
        crit = nn.BCEWithLogitsLoss()

    running_loss, seen = 0.0, 0
    running_correct = 0.0

    it = loader
    if use_tqdm:
        it = tqdm(loader, desc=f"[{desc}] ep{epoch}", leave=True, dynamic_ncols=True)

    t0 = time.time()
    for step, (x, y) in enumerate(it, 1):
        x, y = x.to(device), y.to(device)
        logit = model(x)
        loss = crit(logit, y)

        bs = x.size(0)
        running_loss += loss.item() * bs
        seen += bs

        probs = torch.sigmoid(logit)
        preds = (probs >= thr).float()
        running_correct += (preds == y).float().sum().item()

        avg_loss = running_loss / max(seen, 1)
        avg_acc = running_correct / max(seen, 1)

        if use_tqdm:
            it.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.3f}")

        if step == 1 or step % print_every == 0 or step == len(loader):
            dt = time.time() - t0
            ips = seen / max(dt, 1e-6)
            print(f"[{desc}] ep={epoch} step={step}/{len(loader)} "
                  f"avg_loss={avg_loss:.6f} acc={avg_acc:.4f} thr={thr:.2f} ips={ips:.2f}")

    return avg_loss, avg_acc

In [8]:
# ==========================================
# 7) Threshold search + metric reporting
# ==========================================
@torch.no_grad()
def predict_probs(model, loader, device):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        logit = model(x)
        p = torch.sigmoid(logit).detach().cpu().numpy().reshape(-1)
        probs.append(p)
        ys.append(y.detach().cpu().numpy().reshape(-1))
    return np.concatenate(probs), np.concatenate(ys).astype(int)

def metrics_at_threshold(probs, ys, thr):
    preds = (probs >= thr).astype(int)
    tp = int(((preds==1) & (ys==1)).sum())
    tn = int(((preds==0) & (ys==0)).sum())
    fp = int(((preds==1) & (ys==0)).sum())
    fn = int(((preds==0) & (ys==1)).sum())

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"thr": float(thr), "acc": acc, "precision": prec, "recall": rec, "f1": f1,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def find_best_threshold(probs, ys, mode="f1"):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None
    best_score = -1.0
    for thr in thresholds:
        m = metrics_at_threshold(probs, ys, thr)
        score = m["f1"] if mode == "f1" else m["acc"]
        # tie-break: prefer higher recall
        if (score > best_score) or (score == best_score and best is not None and m["recall"] > best["recall"]):
            best_score = score
            best = m
    return best

def print_metrics(title, m):
    print(f"\n{title}")
    print(f"  thr={m['thr']:.3f} | acc={m['acc']:.4f} f1={m['f1']:.4f} prec={m['precision']:.4f} rec={m['recall']:.4f}")
    print(f"  TP={m['tp']} FP={m['fp']} TN={m['tn']} FN={m['fn']}")

In [9]:
# ==========================================
# 8) STAGE 1: hemorrhage vs non-hem (ALL datasets)
# - Balance train
# - Save best by VAL F1-thresholded (after threshold search each epoch)
#   (This is more stable than raw accuracy when class imbalance exists)
# ==========================================
s1_train = studies_df[studies_df.split=="train"].copy()
s1_val   = studies_df[studies_df.split=="val"].copy()
s1_test  = studies_df[studies_df.split=="test"].copy()

print("S1 train y_hemo:\n", s1_train["y_hemo"].value_counts())
print("S1 val   y_hemo:\n", s1_val["y_hemo"].value_counts())
print("S1 test  y_hemo:\n", s1_test["y_hemo"].value_counts())

def balance_binary(df_in, ycol, seed=42):
    pos = df_in[df_in[ycol]==1]
    neg = df_in[df_in[ycol]==0]
    n = min(len(pos), len(neg))
    pos = pos.sample(n, random_state=seed)
    neg = neg.sample(n, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

s1_train_bal = balance_binary(s1_train, "y_hemo", seed=SEED)

# Background-safe loaders: num_workers=0 avoids Kaggle multiprocessing cleanup bugs
dl_tr = DataLoader(CTStudyDataset(s1_train_bal, "y_hemo", augment=True), batch_size=2, shuffle=True,  num_workers=0, pin_memory=False)
dl_va = DataLoader(CTStudyDataset(s1_val,       "y_hemo", augment=False), batch_size=2, shuffle=False, num_workers=0, pin_memory=False)
dl_te = DataLoader(CTStudyDataset(s1_test,      "y_hemo", augment=False), batch_size=2, shuffle=False, num_workers=0, pin_memory=False)

model_s1 = StudyTransformer(pretrained=True).to(device)
opt_s1 = torch.optim.AdamW(model_s1.parameters(), lr=2e-4, weight_decay=1e-4)
crit_s1 = nn.BCEWithLogitsLoss()

best_val_f1 = -1.0
best_thr_s1 = 0.5

EPOCHS = 5
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_one_epoch(model_s1, dl_tr, opt_s1, device, epoch, crit_s1, desc="S1-train", thr=0.5)
    va_loss, va_acc = eval_one_epoch(model_s1, dl_va, device, epoch, crit_s1, desc="S1-val", thr=0.5)

    # Find best threshold on VAL for this epoch
    p_val, y_val = predict_probs(model_s1, dl_va, device)
    best_epoch = find_best_threshold(p_val, y_val, mode="f1")

    print(f"[S1] epoch {epoch} tr_loss={tr_loss:.4f} tr_acc@0.5={tr_acc:.4f} | "
          f"va_loss={va_loss:.4f} va_acc@0.5={va_acc:.4f} | "
          f"val_best_thr={best_epoch['thr']:.3f} val_best_f1={best_epoch['f1']:.4f}")

    if best_epoch["f1"] > best_val_f1:
        best_val_f1 = best_epoch["f1"]
        best_thr_s1 = best_epoch["thr"]
        torch.save(model_s1.state_dict(), "stage1_best.pt")
        print("  saved -> stage1_best.pt")

# Evaluate Stage 1 on TEST with the VAL-best threshold
model_s1.load_state_dict(torch.load("stage1_best.pt", map_location=device))
p_val, y_val = predict_probs(model_s1, dl_va, device)
best_s1 = find_best_threshold(p_val, y_val, mode="f1")
print_metrics("Stage 1 VAL best", best_s1)

p_test, y_test = predict_probs(model_s1, dl_te, device)
m_test = metrics_at_threshold(p_test, y_test, best_s1["thr"])
print_metrics("Stage 1 TEST using VAL-best thr", m_test)

print("\n[Stage 1] Final chosen threshold:", best_s1["thr"])

S1 train y_hemo:
 y_hemo
1    6413
0    1776
Name: count, dtype: int64
S1 val   y_hemo:
 y_hemo
1    812
0    215
Name: count, dtype: int64
S1 test  y_hemo:
 y_hemo
1    782
0    215
Name: count, dtype: int64


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

[S1-train] ep1:   0%|          | 0/1776 [00:00<?, ?it/s]

[S1-train] ep=1 step=1/1776 avg_loss=0.937237 acc=0.5000 thr=0.50 ips=0.24
[S1-train] ep=1 step=100/1776 avg_loss=0.448823 acc=0.8200 thr=0.50 ips=1.56
[S1-train] ep=1 step=200/1776 avg_loss=0.369002 acc=0.8750 thr=0.50 ips=1.58
[S1-train] ep=1 step=300/1776 avg_loss=0.322411 acc=0.8950 thr=0.50 ips=1.57
[S1-train] ep=1 step=400/1776 avg_loss=0.307141 acc=0.9000 thr=0.50 ips=1.56
[S1-train] ep=1 step=500/1776 avg_loss=0.280431 acc=0.9100 thr=0.50 ips=1.51
[S1-train] ep=1 step=600/1776 avg_loss=0.253528 acc=0.9208 thr=0.50 ips=1.47
[S1-train] ep=1 step=700/1776 avg_loss=0.244513 acc=0.9214 thr=0.50 ips=1.44
[S1-train] ep=1 step=800/1776 avg_loss=0.301457 acc=0.8931 thr=0.50 ips=1.42
[S1-train] ep=1 step=900/1776 avg_loss=0.317665 acc=0.8828 thr=0.50 ips=1.42
[S1-train] ep=1 step=1000/1776 avg_loss=0.342339 acc=0.8700 thr=0.50 ips=1.41
[S1-train] ep=1 step=1100/1776 avg_loss=0.366630 acc=0.8505 thr=0.50 ips=1.42
[S1-train] ep=1 step=1200/1776 avg_loss=0.392016 acc=0.8321 thr=0.50 ips=1.4

[S1-val] ep1:   0%|          | 0/514 [00:00<?, ?it/s]

...

[S1-val] ep=4 step=1/514 avg_loss=0.686318 acc=0.5000 thr=0.50 ips=3.64
[S1-val] ep=4 step=200/514 avg_loss=0.626985 acc=0.6025 thr=0.50 ips=2.72
[S1-val] ep=4 step=400/514 avg_loss=0.616263 acc=0.7450 thr=0.50 ips=3.36
[S1-val] ep=4 step=514/514 avg_loss=0.618706 acc=0.7722 thr=0.50 ips=3.55
[S1] epoch 4 tr_loss=0.5798 tr_acc@0.5=0.6965 | va_loss=0.6187 va_acc@0.5=0.7722 | val_best_thr=0.390 val_best_f1=0.8897
  saved -> stage1_best.pt


[S1-train] ep5:   0%|          | 0/1776 [00:00<?, ?it/s]

[S1-train] ep=5 step=1/1776 avg_loss=0.931048 acc=0.0000 thr=0.50 ips=2.35
[S1-train] ep=5 step=100/1776 avg_loss=0.675615 acc=0.5850 thr=0.50 ips=1.69
[S1-train] ep=5 step=200/1776 avg_loss=0.647229 acc=0.6325 thr=0.50 ips=1.73
[S1-train] ep=5 step=300/1776 avg_loss=0.636269 acc=0.6467 thr=0.50 ips=1.72
[S1-train] ep=5 step=400/1776 avg_loss=0.622556 acc=0.6575 thr=0.50 ips=1.74
[S1-train] ep=5 step=500/1776 avg_loss=0.611057 acc=0.6700 thr=0.50 ips=1.72
[S1-train] ep=5 step=600/1776 avg_loss=0.604334 acc=0.6758 thr=0.50 ips=1.73
[S1-train] ep=5 step=700/1776 avg_loss=0.598053 acc=0.6843 thr=0.50 ips=1.72
[S1-train] ep=5 step=800/1776 avg_loss=0.583796 acc=0.6987 thr=0.50 ips=1.73
[S1-train] ep=5 step=900/1776 avg_loss=0.585787 acc=0.7017 thr=0.50 ips=1.71
[S1-train] ep=5 step=1000/1776 avg_loss=0.587419 acc=0.7025 thr=0.50 ips=1.70
[S1-train] ep=5 step=1100/1776 avg_loss=0.588113 acc=0.7045 thr=0.50 ips=1.70
[S1-train] ep=5 step=1200/1776 avg_loss=0.584777 acc=0.7079 thr=0.50 ips=1.6

[S1-val] ep5:   0%|          | 0/514 [00:00<?, ?it/s]

[S1-val] ep=5 step=1/514 avg_loss=1.645212 acc=0.0000 thr=0.50 ips=3.90
[S1-val] ep=5 step=200/514 avg_loss=1.284562 acc=0.3850 thr=0.50 ips=2.71
[S1-val] ep=5 step=400/514 avg_loss=0.919989 acc=0.6025 thr=0.50 ips=3.28
[S1-val] ep=5 step=514/514 avg_loss=0.851641 acc=0.6465 thr=0.50 ips=3.45
[S1] epoch 5 tr_loss=0.5612 tr_acc@0.5=0.7340 | va_loss=0.8516 va_acc@0.5=0.6465 | val_best_thr=0.010 val_best_f1=0.8831

Stage 1 VAL best
  thr=0.390 | acc=0.8140 f1=0.8889 prec=0.8423 rec=0.9409
  TP=764 FP=143 TN=72 FN=48

Stage 1 TEST using VAL-best thr
  thr=0.390 | acc=0.8014 f1=0.8819 prec=0.8266 rec=0.9450
  TP=739 FP=155 TN=60 FN=43

[Stage 1] Final chosen threshold: 0.39


In [10]:
# ==========================================
# 9) STAGE 2: ischemia vs other (ONLY non-hem)
# - Use Focal+BCE + pos_weight
# - Save best by VAL F1 (after threshold search)
# ==========================================
base = studies_df[studies_df["y_hemo"]==0].copy()
s2_train = base[base.split=="train"].copy()
s2_val   = base[base.split=="val"].copy()
s2_test  = base[base.split=="test"].copy()

print("S2 train y_isc:\n", s2_train["y_isc"].value_counts())
print("S2 val   y_isc:\n", s2_val["y_isc"].value_counts())
print("S2 test  y_isc:\n", s2_test["y_isc"].value_counts())

pos = int((s2_train["y_isc"]==1).sum())
neg = int((s2_train["y_isc"]==0).sum())
pos_weight = torch.tensor([neg / max(pos, 1)], device=device)
crit_s2 = FocalBCEWithLogits(pos_weight=pos_weight, gamma=2.0)
print("Stage 2 pos_weight:", float(pos_weight.item()))

dl_tr2 = DataLoader(CTStudyDataset(s2_train, "y_isc", augment=True), batch_size=2, shuffle=True,  num_workers=0, pin_memory=False)
dl_va2 = DataLoader(CTStudyDataset(s2_val,   "y_isc", augment=False), batch_size=2, shuffle=False, num_workers=0, pin_memory=False)
dl_te2 = DataLoader(CTStudyDataset(s2_test,  "y_isc", augment=False), batch_size=2, shuffle=False, num_workers=0, pin_memory=False)

model_s2 = StudyTransformer(pretrained=True).to(device)
opt_s2 = torch.optim.AdamW(model_s2.parameters(), lr=2e-4, weight_decay=1e-4)

best_val_f1 = -1.0

EPOCHS = 5
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_one_epoch(model_s2, dl_tr2, opt_s2, device, epoch, crit_s2, desc="S2-train", thr=0.5)
    va_loss, va_acc = eval_one_epoch(model_s2, dl_va2, device, epoch, crit_s2, desc="S2-val", thr=0.5)

    p_val, y_val = predict_probs(model_s2, dl_va2, device)
    best_epoch = find_best_threshold(p_val, y_val, mode="f1")

    print(f"[S2] epoch {epoch} tr_loss={tr_loss:.4f} tr_acc@0.5={tr_acc:.4f} | "
          f"va_loss={va_loss:.4f} va_acc@0.5={va_acc:.4f} | "
          f"val_best_thr={best_epoch['thr']:.3f} val_best_f1={best_epoch['f1']:.4f}")

    if best_epoch["f1"] > best_val_f1:
        best_val_f1 = best_epoch["f1"]
        torch.save(model_s2.state_dict(), "stage2_best.pt")
        print("  saved -> stage2_best.pt")

# Evaluate Stage 2 on TEST with the VAL-best threshold
model_s2.load_state_dict(torch.load("stage2_best.pt", map_location=device))
p_val, y_val = predict_probs(model_s2, dl_va2, device)
best_s2 = find_best_threshold(p_val, y_val, mode="f1")
print_metrics("Stage 2 VAL best", best_s2)

p_test, y_test = predict_probs(model_s2, dl_te2, device)
m_test = metrics_at_threshold(p_test, y_test, best_s2["thr"])
print_metrics("Stage 2 TEST using VAL-best thr", m_test)

print("\n[Stage 2] Final chosen threshold:", best_s2["thr"])

S2 train y_isc:
 y_isc
0    987
1    789
Name: count, dtype: int64
S2 val   y_isc:
 y_isc
0    117
1     98
Name: count, dtype: int64
S2 test  y_isc:
 y_isc
0    120
1     95
Name: count, dtype: int64
Stage 2 pos_weight: 1.250950574874878


[S2-train] ep1:   0%|          | 0/888 [00:00<?, ?it/s]

[S2-train] ep=1 step=1/888 avg_loss=0.188441 acc=0.5000 thr=0.50 ips=0.97
[S2-train] ep=1 step=100/888 avg_loss=0.229932 acc=0.7500 thr=0.50 ips=1.36
[S2-train] ep=1 step=200/888 avg_loss=0.180547 acc=0.7775 thr=0.50 ips=1.36
[S2-train] ep=1 step=300/888 avg_loss=0.179483 acc=0.7700 thr=0.50 ips=1.37
[S2-train] ep=1 step=400/888 avg_loss=0.155497 acc=0.7963 thr=0.50 ips=1.38
[S2-train] ep=1 step=500/888 avg_loss=0.146527 acc=0.8200 thr=0.50 ips=1.38
[S2-train] ep=1 step=600/888 avg_loss=0.140472 acc=0.8233 thr=0.50 ips=1.38
[S2-train] ep=1 step=700/888 avg_loss=0.134164 acc=0.8314 thr=0.50 ips=1.38
[S2-train] ep=1 step=800/888 avg_loss=0.136687 acc=0.8306 thr=0.50 ips=1.37
[S2-train] ep=1 step=888/888 avg_loss=0.135582 acc=0.8283 thr=0.50 ips=1.36


[S2-val] ep1:   0%|          | 0/108 [00:00<?, ?it/s]

[S2-val] ep=1 step=1/108 avg_loss=0.148073 acc=0.5000 thr=0.50 ips=2.57
[S2-val] ep=1 step=108/108 avg_loss=0.058334 acc=0.9302 thr=0.50 ips=2.15
[S2] epoch 1 tr_loss=0.1356 tr_acc@0.5=0.8283 | va_loss=0.0583 va_acc@0.5=0.9302 | val_best_thr=0.300 val_best_f1=0.9548
  saved -> stage2_best.pt


...

[S2-val] ep=4 step=1/108 avg_loss=0.395879 acc=0.5000 thr=0.50 ips=3.85
[S2-val] ep=4 step=108/108 avg_loss=0.265341 acc=0.4930 thr=0.50 ips=2.21
[S2] epoch 4 tr_loss=0.1815 tr_acc@0.5=0.6109 | va_loss=0.2653 va_acc@0.5=0.4930 | val_best_thr=0.010 val_best_f1=0.6262


[S2-train] ep5:   0%|          | 0/888 [00:00<?, ?it/s]

[S2-train] ep=5 step=1/888 avg_loss=0.064812 acc=1.0000 thr=0.50 ips=2.08
[S2-train] ep=5 step=100/888 avg_loss=0.162912 acc=0.7650 thr=0.50 ips=1.34
[S2-train] ep=5 step=200/888 avg_loss=0.159054 acc=0.7275 thr=0.50 ips=1.38
[S2-train] ep=5 step=300/888 avg_loss=0.152969 acc=0.7517 thr=0.50 ips=1.35
[S2-train] ep=5 step=400/888 avg_loss=0.149482 acc=0.7538 thr=0.50 ips=1.37
[S2-train] ep=5 step=500/888 avg_loss=0.147764 acc=0.7710 thr=0.50 ips=1.36
[S2-train] ep=5 step=600/888 avg_loss=0.156600 acc=0.7300 thr=0.50 ips=1.37
[S2-train] ep=5 step=700/888 avg_loss=0.162634 acc=0.6986 thr=0.50 ips=1.37
[S2-train] ep=5 step=800/888 avg_loss=0.166349 acc=0.6744 thr=0.50 ips=1.37
[S2-train] ep=5 step=888/888 avg_loss=0.165967 acc=0.6768 thr=0.50 ips=1.37


[S2-val] ep5:   0%|          | 0/108 [00:00<?, ?it/s]

[S2-val] ep=5 step=1/108 avg_loss=0.180632 acc=0.5000 thr=0.50 ips=3.99
[S2-val] ep=5 step=108/108 avg_loss=0.186462 acc=0.6744 thr=0.50 ips=2.17
[S2] epoch 5 tr_loss=0.1660 tr_acc@0.5=0.6768 | va_loss=0.1865 va_acc@0.5=0.6744 | val_best_thr=0.370 val_best_f1=0.7273

Stage 2 VAL best
  thr=0.270 | acc=0.9628 f1=0.9596 prec=0.9500 rec=0.9694
  TP=95 FP=5 TN=112 FN=3

Stage 2 TEST using VAL-best thr
  thr=0.270 | acc=0.9163 f1=0.9100 prec=0.8667 rec=0.9579
  TP=91 FP=14 TN=106 FN=4

[Stage 2] Final chosen threshold: 0.27
